# August 2026 verification · **07 — Gold: Cancer Waiting Times**

**The claims under test (article section 6):**

* 62-day performance **71.9%** against an **85%** standard, "improved year on year and still thirteen points short";
* 28-day Faster Diagnosis **78.6%** against an **80%** operational target, "a target missed by under two points reads as near-success and still means tens of thousands told late".

The gate (04) already rebuilt 71.9% and 78.6% exactly. Here I test the
claims built **on top of** those numbers:

1. the standards themselves, read from NHS England's own file;
2. the gap to standard, in points **and in people**;
3. year on year, like for like (provisional against provisional) and against the revised (final) August 2025;
4. breach counts and how late the late patients were;
5. the spread between providers: how much the national figure hides.

All figures are provider basis, compliance recomputed as Within / Total.
August 2026 is **provisional**. Outputs go to `data/gold/07_cancer/`.

In [1]:
import math
import re
import sys
from pathlib import Path

import numpy as np
import openpyxl
import pandas as pd
import pypdf

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import BRONZE_DIR, GOLD_DIR, SILVER_DIR  # noqa: E402

OUT = GOLD_DIR / "07_cancer"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
CWT = BRONZE_DIR / "nhse_cwt_2026-08"
MEASURES = ["62D", "FDS"]
NAMES = {"62D": "62-day combined", "FDS": "28-day Faster Diagnosis"}

## Step 0 — The gate must have passed, including the cancer checks

In [2]:
gate = pd.read_csv(GOLD_DIR / "00_reconciliation" / "reconciliation.csv")
assert gate["pass"].all(), f"reconciliation gate has {(~gate['pass']).sum()} failing checks: run 04 first"
assert (gate.gate == "C").sum() >= 14, "the cancer checks are missing from the gate"
cwt = pd.read_parquet(SILVER_DIR / "cwt" / "cwt_provider_headline.parquet")
eng_row = pd.read_parquet(SILVER_DIR / "cwt" / "cwt_england_row.parquet")
prov25 = pd.read_parquet(SILVER_DIR / "cwt" / "cwt_2025-08_provisional.parquet")
print(f"gate: {len(gate)} of {len(gate)} checks passed")

gate: 51 of 51 checks passed


## Step 1 — The standards, from NHS England's own file

The national time series states each standard in its header row. I read
the standard from there rather than typing 85 and 80 in.

In [3]:
nts = openpyxl.load_workbook(CWT / "CWT-CRS-National-Time-Series-Oct-2009-Aug-2026-Provisional.xlsx",
                             read_only=True, data_only=True)["Monthly Performance"]
nts.reset_dimensions()
trows = list(nts.iter_rows(values_only=True))
TITLES = {"FDS": "Four Week (28 days)", "31D": "One Month (31-day)", "62D": "Two Month (62-day)"}
block = {m: next(i for i, x in enumerate(trows[1]) if x and str(x).startswith(t)) for m, t in TITLES.items()}
STANDARD_TEXT = {m: str(trows[2][c]) for m, c in block.items()}
STANDARD = {m: float(re.search(r"= (\d+)%", STANDARD_TEXT[m]).group(1)) / 100 for m in block}
NTS = {(m, r[c - 1].strftime("%Y-%m")): (r[c], r[c + 1]) for r in trows[4:] for m, c in block.items()
       if hasattr(r[c - 1], "strftime")}
print(STANDARD_TEXT)
assert STANDARD["62D"] == 0.85 and STANDARD["FDS"] == 0.80

{'FDS': 'Operational Standard = 80% from Q1 2026/27', '31D': 'Operational Standard = 96% from Q3 2023/24', '62D': 'Operational Standard = 85% from Q3 2023/24'}


## Step 2 — August 2026 against the standard, in points and in people

A percentage gap hides a count. For each standard I compute:

* **late**: patients outside the standard (`After`);
* **shortfall**: how many more patients would have needed to be on time for the standard to be met, `ceil(standard × Total) − Within`;
* **late but allowed**: patients the standard itself permits to be late, `Total − ceil(standard × Total)`.

Late = shortfall + late-but-allowed. That split is what tests "tens of
thousands told late".

In [4]:
nat = cwt.groupby(["measure", "period"])[["total", "within", "after"]].sum()
nat["pct"] = 100 * nat.within / nat.total

def vs_standard(m: str, p: str) -> dict:
    t, w, a = nat.loc[(m, p), ["total", "within", "after"]]
    need = math.ceil(STANDARD[m] * t)
    return {"measure": m, "period": p, "total": t, "within": w, "late": a, "pct": 100 * w / t,
            "standard_pct": 100 * STANDARD[m], "gap_pp": 100 * STANDARD[m] - 100 * w / t,
            "shortfall_people": max(need - w, 0), "late_but_allowed": t - need}

std = pd.DataFrame([vs_standard(m, p) for m in MEASURES for p in ["2025-08", "2026-07", "2026-08"]])
assert (std.shortfall_people + std.late_but_allowed - std.late).abs().max() < 1e-6
std.to_csv(OUT / "vs_standard.csv", index=False)
std

,measure,period,total,within,late,pct,standard_pct,gap_pp,shortfall_people,late_but_allowed
0,62D,2025-08,"27,533.00","19,090.00","8,443.00",69.33,85.00,15.67,"4,314.00","4,129.00"
1,62D,2026-07,"31,519.00","22,428.00","9,091.00",71.16,85.00,13.84,"4,364.00","4,727.00"
2,62D,2026-08,"27,909.00","20,066.00","7,843.00",71.90,85.00,13.10,"3,657.00","4,186.00"
3,FDS,2025-08,"273,040.00","203,750.00","69,290.00",74.62,80.00,5.38,"14,682.00","54,608.00"
4,FDS,2026-07,"308,370.00","244,594.00","63,776.00",79.32,80.00,0.68,"2,102.00","61,674.00"
5,FDS,2026-08,"273,664.00","214,991.00","58,673.00",78.56,80.00,1.44,"3,941.00","54,732.00"


## Step 3 — Year on year, two ways

August 2026 is provisional. August 2025 exists in two vintages: as first
published (provisional, 9 October 2025) and as revised (the 2025-26 Final
file). The like-for-like comparison is **provisional against provisional**.
Comparing provisional August 2026 with final August 2025 mixes vintages.
I show both, plus how much August 2025 moved when it was finalised. If the
revision is similar every year, August 2026 will probably move the same way.

In [5]:
# Silver keeps the 'ALL ENGLISH PROVIDERS' row beside the trusts; I sum trusts only.
p25 = prov25[prov25.org_name.ne("ALL ENGLISH PROVIDERS")].groupby("measure")[["total", "within", "after"]].sum()
yoy = []
for m in MEASURES:
    a26 = nat.loc[(m, "2026-08")]
    f25 = nat.loc[(m, "2025-08")]
    v25 = p25.loc[m]
    # The provisional parse must agree with NHS England's provisional national series.
    assert (int(v25.total), int(v25.within)) == tuple(int(x) for x in NTS[(m, "2025-08")]), \
        f"{m}: Aug-25 provisional file != national time series (provisional)"
    for label, b in [("vs Aug 2025 provisional (like for like)", v25), ("vs Aug 2025 final (mixed vintages)", f25)]:
        yoy.append({"measure": m, "comparison": label,
                    "aug_2025_pct": 100 * b.within / b.total, "aug_2026_pct": 100 * a26.within / a26.total,
                    "change_pp": 100 * (a26.within / a26.total - b.within / b.total),
                    "aug_2025_total": b.total, "aug_2026_total": a26.total,
                    "late_2025": b.after, "late_2026": a26.after, "late_change": a26.after - b.after})
    yoy.append({"measure": m, "comparison": "Aug 2025 revision (provisional -> final)",
                "aug_2025_pct": 100 * v25.within / v25.total, "aug_2026_pct": 100 * f25.within / f25.total,
                "change_pp": 100 * (f25.within / f25.total - v25.within / v25.total),
                "aug_2025_total": v25.total, "aug_2026_total": f25.total,
                "late_2025": v25.after, "late_2026": f25.after, "late_change": f25.after - v25.after})
yoy = pd.DataFrame(yoy)
yoy.to_csv(OUT / "year_on_year.csv", index=False)
yoy

,measure,comparison,aug_2025_pct,aug_2026_pct,change_pp,aug_2025_total,aug_2026_total,late_2025,late_2026,late_change
0,62D,vs Aug 2025 provisional (like for like),69.11,71.90,2.78,"26,906.00","27,909.00","8,310.00","7,843.00",-467.00
1,62D,vs Aug 2025 final (mixed vintages),69.33,71.90,2.56,"27,533.00","27,909.00","8,443.00","7,843.00",-600.00
2,62D,Aug 2025 revision (provisional -> final),69.11,69.33,0.22,"26,906.00","27,533.00","8,310.00","8,443.00",133.00
3,FDS,vs Aug 2025 provisional (like for like),74.61,78.56,3.95,"267,937.00","273,664.00","68,023.00","58,673.00","-9,350.00"
4,FDS,vs Aug 2025 final (mixed vintages),74.62,78.56,3.94,"273,040.00","273,664.00","69,290.00","58,673.00","-10,617.00"
5,FDS,Aug 2025 revision (provisional -> final),74.61,74.62,0.01,"267,937.00","273,040.00","68,023.00","69,290.00","1,267.00"


### The release's 12-month averages

The release says FDS "averaged 77.3%" and 62-day "70.2%" from September 2025
to August 2026. It doesn't say whether that is the mean of monthly
percentages or the pooled rate, so I compute both and see which one it is.
This also tests the mixed vintages the release itself uses: September-March final, April-August provisional.

In [6]:
rel = re.sub(r"\s+", " ", " ".join(p.extract_text() for p in pypdf.PdfReader(
    CWT / "Cancer-Waiting-Times-Statistical-Release-August-2026-Provider-based-Provisional.pdf").pages))
REL_AVG = {
    "FDS": float(re.search(r"Performance time series for FDS .*? averaged ([\d.]+)%", rel).group(1)),
    "62D": float(re.search(r"Performance time series for the 62-day Combined standard .*? averaged ([\d.]+)%", rel).group(1)),
}
months12 = pd.period_range("2025-09", "2026-08", freq="M").strftime("%Y-%m")
avg = []
for m in MEASURES:
    x = nat.loc[m].loc[months12]
    avg.append({"measure": m, "release_says": REL_AVG[m], "mean_of_monthly_pct": x.pct.mean(),
                "pooled_pct": 100 * x.within.sum() / x.total.sum()})
avg = pd.DataFrame(avg)
# I list every method that reproduces the published figure. If both do, the
# release's method can't be told apart at one decimal place, and I say so
# rather than pick one.
avg["methods_that_match"] = avg.apply(lambda r: ", ".join(
    [n for n, v in (("mean of monthly %", r.mean_of_monthly_pct), ("pooled", r.pooled_pct)) if round(v, 1) == r.release_says])
    or "neither", axis=1)
assert (avg.methods_that_match != "neither").all(), "a 12-month average in the release does not reproduce"
avg.to_csv(OUT / "twelve_month_average.csv", index=False)
avg

,measure,release_says,mean_of_monthly_pct,pooled_pct,methods_that_match
0,62D,70.20,70.20,70.21,"mean of monthly %, pooled"
1,FDS,77.30,77.35,77.35,"mean of monthly %, pooled"


## Step 4 — How late were the late patients?

The bands tell me how late, not just whether late. Silver checked they add
up to Total. Here I split the late patients only.

In [7]:
LATE_BANDS = {
    "FDS": {"29-42 days": "In_29_to_42_days", "43-62 days": "In_43_to_62_days", "over 62 days": "After_62_days"},
    "62D": {"63-76 days": "In_63_to_76_days", "77-90 days": "In_77_to_90_days",
            "91-104 days": "In_91_to_104_days", "over 104 days": "After_104_days"},
}
depth = []
for m, bands in LATE_BANDS.items():
    for p in ["2025-08", "2026-08"]:
        e = eng_row[(eng_row.measure == m) & (eng_row.period == p)].iloc[0]
        late_sum = sum(e[c] for c in bands.values())
        assert abs(late_sum - e.after) < 1e-6, f"{m} {p}: late bands != After"
        for label, c in bands.items():
            depth.append({"measure": m, "period": p, "band": label, "patients": e[c], "share_of_late": e[c] / e.after})
depth = pd.DataFrame(depth)
depth.to_csv(OUT / "breach_depth.csv", index=False)
depth.pivot_table(index=["measure", "band"], columns="period", values=["patients", "share_of_late"], sort=False)

patients           share_of_late        
period                  2025-08   2026-08       2025-08 2026-08
measure band                                                   
FDS     29-42 days    36,605.00 31,267.00          0.53    0.53
        43-62 days    20,490.00 17,298.00          0.30    0.29
        over 62 days  12,195.00 10,108.00          0.18    0.17
62D     63-76 days     2,761.00  2,547.00          0.33    0.32
        77-90 days     1,973.00  1,885.00          0.23    0.24
        91-104 days    1,273.00  1,254.00          0.15    0.16
        over 104 days  2,436.00  2,157.00          0.29    0.28

## Step 5 — Provider spread

One month at one trust can be a handful of patients, especially for 62-day.
So I do three things:

* set a **minimum volume** (at least 20 patients in the period) and report how many trusts that drops;
* repeat on **June-August pooled**, which is steadier;
* give each trust a **Wilson 95% interval**, and count the trusts whose *upper* bound is below the standard, i.e. clearly below it rather than unlucky in a small month.

Trusts are identified by ODS code. The 2026 periods use a single
provisional vintage. For the pooled year-on-year comparison, June-August
2025 comes from the Final file, so that comparison mixes vintages like the
national one does.

In [8]:
MIN_N = 20

def wilson(w, n, z=1.96):
    p = w / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

def trust_table(m: str, months: list[str]) -> pd.DataFrame:
    x = cwt[(cwt.measure == m) & cwt.period.isin(months)]
    g = x.groupby("org_code").agg(org_name=("org_name", "last"), total=("total", "sum"),
                                  within=("within", "sum"), after=("after", "sum"), months=("period", "nunique"))
    g = g[g.total > 0]
    g["pct"] = 100 * g.within / g.total
    ci = g.apply(lambda r: wilson(r.within, r.total), axis=1, result_type="expand")
    g["ci_lo"], g["ci_hi"] = 100 * ci[0], 100 * ci[1]
    return g

spread, tables = [], {}
for m in MEASURES:
    for label, months in [("Aug 2026", ["2026-08"]), ("Jun-Aug 2026", ["2026-06", "2026-07", "2026-08"]),
                          ("Aug 2025", ["2025-08"]), ("Jun-Aug 2025", ["2025-06", "2025-07", "2025-08"])]:
        g = trust_table(m, months)
        tables[(m, label)] = g
        k = g[g.total >= MIN_N]
        s = 100 * STANDARD[m]
        late_sorted = g.after.sort_values(ascending=False)
        spread.append({
            "measure": m, "period": label, "trusts": len(g), f"trusts_n>={MIN_N}": len(k),
            "national_pct": 100 * g.within.sum() / g.total.sum(),
            "min": k.pct.min(), "p10": k.pct.quantile(.1), "median": k.pct.median(),
            "p90": k.pct.quantile(.9), "max": k.pct.max(), "p90_minus_p10": k.pct.quantile(.9) - k.pct.quantile(.1),
            "meeting_standard": int((k.pct >= s).sum()),
            "clearly_below_standard": int((k.ci_hi < s).sum()),
            "share_of_late_at_top20_trusts": late_sorted.head(20).sum() / g.after.sum(),
        })
spread = pd.DataFrame(spread)
spread.to_csv(OUT / "provider_spread.csv", index=False)
spread

,measure,period,trusts,trusts_n>=20,national_pct,min,p10,median,p90,max,p90_minus_p10,meeting_standard,clearly_below_standard,share_of_late_at_top20_trusts
0,62D,Aug 2026,143,125,71.90,51.75,61.62,73.84,83.79,96.23,22.17,10,97,0.38
1,62D,Jun-Aug 2026,152,136,70.96,51.79,60.70,73.07,84.70,100.00,23.99,13,109,0.38
2,62D,Aug 2025,149,127,69.33,43.18,59.34,71.03,83.51,95.45,24.17,11,101,0.38
3,62D,Jun-Aug 2025,155,135,68.73,33.64,58.76,70.22,83.77,100.00,25.01,10,112,0.37
4,FDS,Aug 2026,140,135,78.56,28.00,71.37,80.55,87.77,100.00,16.40,74,48,0.36
5,FDS,Jun-Aug 2026,141,140,78.96,4.76,71.32,80.77,87.01,100.00,15.69,78,53,0.35
6,FDS,Aug 2025,142,134,74.62,0.00,64.52,75.60,84.75,97.03,20.23,38,83,0.33
7,FDS,Jun-Aug 2025,145,139,75.99,21.88,68.81,77.39,85.48,97.85,16.68,45,82,0.34


In [9]:
# Best and worst five trusts, June-August 2026 pooled, at least MIN_N patients.
for m in MEASURES:
    k = tables[(m, "Jun-Aug 2026")]
    k = k[k.total >= MIN_N].sort_values("pct")
    k.to_csv(OUT / f"trusts_{m}_jun_aug_2026.csv")
    print(f"== {NAMES[m]}, Jun-Aug 2026 pooled, n >= {MIN_N}: lowest and highest five")
    print(pd.concat([k.head(5), k.tail(5)])[["org_name", "total", "pct", "ci_lo", "ci_hi"]].round(1).to_string(), "\n")

== 62-day combined, Jun-Aug 2026 pooled, n >= 20: lowest and highest five
                                                                org_name    total    pct  ci_lo  ci_hi
org_code                                                                                              
RHQ                    SHEFFIELD TEACHING HOSPITALS NHS FOUNDATION TRUST 1,173.00  51.80  48.90  54.60
RAJ                             MID AND SOUTH ESSEX NHS FOUNDATION TRUST 1,533.00  52.80  50.30  55.30
RWA                         HULL UNIVERSITY TEACHING HOSPITALS NHS TRUST 1,082.50  53.60  50.60  56.60
RWP                             WORCESTERSHIRE ACUTE HOSPITALS NHS TRUST 1,104.00  55.20  52.20  58.10
RRJ                  THE ROYAL ORTHOPAEDIC HOSPITAL NHS FOUNDATION TRUST    45.00  55.60  41.20  69.10
RQ3               BIRMINGHAM WOMEN'S AND CHILDREN'S NHS FOUNDATION TRUST    35.00  91.40  77.60  97.00
RAN                        ROYAL NATIONAL ORTHOPAEDIC HOSPITAL NHS TRUST    67.50  92.60  83.80  96.80

In [10]:
# Trust-level year on year: Jun-Aug 2025 (final) vs Jun-Aug 2026 (provisional), trusts with n >= MIN_N in both.
tyoy = []
for m in MEASURES:
    a, b = tables[(m, "Jun-Aug 2025")], tables[(m, "Jun-Aug 2026")]
    j = a[a.total >= MIN_N][["pct"]].join(b[b.total >= MIN_N][["pct"]], lsuffix="_2025", rsuffix="_2026", how="inner")
    j["change_pp"] = j.pct_2026 - j.pct_2025
    tyoy.append({"measure": m, "trusts_in_both": len(j), "improved": int((j.change_pp > 0).sum()),
                 "worsened": int((j.change_pp < 0).sum()), "median_change_pp": j.change_pp.median(),
                 "p10_change_pp": j.change_pp.quantile(.1), "p90_change_pp": j.change_pp.quantile(.9)})
tyoy = pd.DataFrame(tyoy)
tyoy.to_csv(OUT / "trust_year_on_year.csv", index=False)
tyoy

,measure,trusts_in_both,improved,worsened,median_change_pp,p10_change_pp,p90_change_pp
0,62D,132,82,50,1.43,-5.72,9.90
1,FDS,134,93,41,2.78,-5.16,9.89


## Step 6 — Verdict, computed

In [11]:
S = std.set_index(["measure", "period"])
Y = yoy.set_index(["measure", "comparison"])
SP = spread.set_index(["measure", "period"])
for m in MEASURES:
    a = S.loc[(m, "2026-08")]
    print(f"{NAMES[m]} Aug 2026 (provisional): {a.pct:.2f}% vs {a.standard_pct:.0f}% standard -> {a.gap_pp:.2f} pp short; "
          f"{a.late:,.0f} late, of whom {a.late_but_allowed:,.0f} the standard itself allows; "
          f"{a.shortfall_people:,.0f} more on time would have met it.")
    ll = Y.loc[(m, "vs Aug 2025 provisional (like for like)")]
    mx = Y.loc[(m, "vs Aug 2025 final (mixed vintages)")]
    rv = Y.loc[(m, "Aug 2025 revision (provisional -> final)")]
    print(f"   year on year: {ll.change_pp:+.2f} pp like for like (from {ll.aug_2025_pct:.1f}%); "
          f"{mx.change_pp:+.2f} pp against final; Aug 2025 moved {rv.change_pp:+.2f} pp and "
          f"{rv.aug_2026_total - rv.aug_2025_total:+,.0f} patients on finalisation.")
    sp = SP.loc[(m, "Jun-Aug 2026")]
    print(f"   providers (Jun-Aug 2026, n>={MIN_N}): {sp[f'trusts_n>={MIN_N}']:.0f} trusts, p10 {sp.p10:.1f}% to p90 {sp.p90:.1f}% "
          f"(range {sp['min']:.1f}-{sp['max']:.1f}%); {sp.meeting_standard:.0f} meet the standard, "
          f"{sp.clearly_below_standard:.0f} are clearly below it; top 20 trusts hold {sp.share_of_late_at_top20_trusts:.0%} of late patients.")
d62 = S.loc[("62D", "2026-08")]
print(f"\nClaim checks: '13 points short' -> {d62.gap_pp:.1f}; 'improved year on year' -> "
      f"{'yes' if Y.loc[('62D', 'vs Aug 2025 provisional (like for like)')].change_pp > 0 else 'NO'}; "
      f"FDS 'missed by under two points' -> {S.loc[('FDS', '2026-08')].gap_pp:.2f}; "
      f"'tens of thousands told late' -> {S.loc[('FDS', '2026-08')].late:,.0f} late, but only "
      f"{S.loc[('FDS', '2026-08')].shortfall_people:,.0f} beyond what the 80% standard permits.")

62-day combined Aug 2026 (provisional): 71.90% vs 85% standard -> 13.10 pp short; 7,843 late, of whom 4,186 the standard itself allows; 3,657 more on time would have met it.
   year on year: +2.78 pp like for like (from 69.1%); +2.56 pp against final; Aug 2025 moved +0.22 pp and +627 patients on finalisation.
   providers (Jun-Aug 2026, n>=20): 136 trusts, p10 60.7% to p90 84.7% (range 51.8-100.0%); 13 meet the standard, 109 are clearly below it; top 20 trusts hold 38% of late patients.
28-day Faster Diagnosis Aug 2026 (provisional): 78.56% vs 80% standard -> 1.44 pp short; 58,673 late, of whom 54,732 the standard itself allows; 3,941 more on time would have met it.
   year on year: +3.95 pp like for like (from 74.6%); +3.94 pp against final; Aug 2025 moved +0.01 pp and +5,103 patients on finalisation.
   providers (Jun-Aug 2026, n>=20): 140 trusts, p10 71.3% to p90 87.0% (range 4.8-100.0%); 78 meet the standard, 53 are clearly below it; top 20 trusts hold 35% of late patients.

Claim 